# Language confound (minimal)

Trimmed-down rerun of `language_confound_analysis.ipynb`: reuses the
persisted `vectorizer.joblib`/`model.joblib` from `train.ipynb` (no
refitting), and adds one trivial baseline -- predict the majority
lecturer for a sentence's language alone, zero word content used -- to
see how much of the model's accuracy language alone could explain.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import joblib
from sklearn.metrics import accuracy_score

data = pd.read_csv('../sentences_top100.csv').dropna(subset=['content'])
data = data[data['content'].str.strip() != ''].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    data['content'], data['lecturer_id'], test_size=0.2, random_state=42, stratify=data['lecturer_id'],
)
lang_train = data['language'].iloc[y_train.index]
lang_test = data['language'].iloc[y_test.index]

vectorizer = joblib.load('vectorizer.joblib')
clf = joblib.load('model.joblib')
y_pred = clf.predict(vectorizer.transform(X_test))

majority_baseline_accuracy = y_test.value_counts(normalize=True).iloc[0]
model_accuracy = accuracy_score(y_test, y_pred)
model_accuracy

0.44654699827341227

In [3]:
# language-only baseline: majority lecturer per language, learned from train
lang_to_majority = y_train.groupby(lang_train).agg(lambda s: s.value_counts().idxmax())
y_pred_lang = lang_test.map(lang_to_majority)
language_baseline_accuracy = accuracy_score(y_test, y_pred_lang)
language_baseline_accuracy

0.20131132657752332

In [ ]:
share_explained = (language_baseline_accuracy - majority_baseline_accuracy) / (model_accuracy - majority_baseline_accuracy)

comparison = pd.DataFrame([
    {'metric': 'majority-class baseline accuracy', 'value': majority_baseline_accuracy},
    {'metric': 'language-only baseline accuracy', 'value': language_baseline_accuracy},
    {'metric': 'model accuracy', 'value': model_accuracy},
    {'metric': 'share of model lift explained by language', 'value': share_explained},
])
comparison.to_csv('language_confound.csv', index=False)
comparison

,metric,value
0,majority-class baseline accuracy,0.176267
1,language-only baseline accuracy,0.201311
2,model accuracy,0.446547
3,share of model lift explained by language,0.092660
